# Notebook 3: Failure Pattern Analysis
## When Do Models Fail? Discovering Hidden Failure Patterns

This is the core analysis notebook covering:
1. Defining failure cases (top 15% prediction errors)
2. Temporal failure patterns (hour, day, month)
3. Spatial failure patterns (which zones fail most)
4. Failure by demand level
5. Cross-model failure comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_ingestion import load_config, download_zone_lookup
from src.models import run_all_models
from src.failure_analysis import (
    run_failure_analysis, compute_errors,
    analyze_temporal_failures, analyze_spatial_failures,
    analyze_demand_level_failures, cross_model_failure_comparison
)
from src.visualizations import (
    plot_failure_rate_by_hour, plot_failure_rate_by_day,
    plot_error_distribution, plot_failure_heatmap,
    plot_top_failure_zones, plot_demand_vs_error,
    plot_monthly_error_trend
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

config = load_config('../configs/config.yaml')
zone_lookup = download_zone_lookup(config)
print('Ready.')

## 1. Load Models & Run Failure Analysis

If you already ran Notebook 2, you can load saved predictions.
Otherwise, retrain models here.

In [ ]:
# Option A: Retrain models (if not already trained)
results = run_all_models(config)

# Run failure analysis
analysis = run_failure_analysis(results, config, zone_lookup)

## 2. Error Distribution

In [ ]:
# Error distributions with failure thresholds
plot_error_distribution(analysis['errors'], config)

# Summary stats per model
for model_name, errors_df in analysis['errors'].items():
    failures = errors_df[errors_df['is_failure'] == 1]
    print(f'\n{model_name}:')
    print(f'  Total predictions: {len(errors_df):,}')
    print(f'  Failure cases: {len(failures):,}')
    print(f'  Mean error (all): {errors_df["abs_error"].mean():.4f}')
    print(f'  Mean error (failures): {failures["abs_error"].mean():.4f}')
    print(f'  Max error: {errors_df["abs_error"].max():.2f}')

## 3. Temporal Failure Patterns

In [ ]:
# Failure rate by hour of day
plot_failure_rate_by_hour(analysis['temporal'], config)

# Which hours have disproportionate failures?
for model_name, temporal in analysis['temporal'].items():
    hourly = temporal['hourly']
    high_failure_hours = hourly[hourly['failure_rate'] > 20]
    if len(high_failure_hours) > 0:
        print(f'\n{model_name} - Hours with >20% failure rate:')
        print(high_failure_hours[['failure_rate', 'mean_abs_error']].to_string())

In [ ]:
# Failure rate by day of week
plot_failure_rate_by_day(analysis['temporal'], config)

# Weekend vs weekday failure comparison
for model_name, temporal in analysis['temporal'].items():
    print(f'\n{model_name} - Weekend vs Weekday:')
    print(pd.DataFrame(temporal['weekend']).to_string(index=False))

In [ ]:
# Failure heatmaps (hour x day)
for model_name, errors_df in analysis['errors'].items():
    plot_failure_heatmap(errors_df, model_name, config)
    
    # Show the heatmap inline too
    dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    pivot = errors_df.groupby(['day_of_week', 'hour'])['is_failure'].mean().reset_index()
    hm = pivot.pivot(index='day_of_week', columns='hour', values='is_failure') * 100
    
    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(hm, cmap='RdYlGn_r', annot=False, ax=ax,
                xticklabels=range(24), yticklabels=dow_names,
                cbar_kws={'label': 'Failure Rate (%)'}, vmin=0, vmax=40)
    ax.set_title(f'{model_name.replace("_", " ").title()} - Failure Rate Heatmap')
    plt.tight_layout()
    plt.show()

In [ ]:
# Monthly error trends
plot_monthly_error_trend(analysis['errors'], config)

# Underprediction vs Overprediction
for model_name, temporal in analysis['temporal'].items():
    d = temporal['direction']
    print(f'\n{model_name} - Error Direction in Failures:')
    print(f'  Underpredictions: {d["underpredictions"]:,} ({d["under_pct"]:.1f}%)')
    print(f'  Overpredictions: {d["overpredictions"]:,} ({100 - d["under_pct"]:.1f}%)')

## 4. Spatial Failure Patterns

In [ ]:
# Top failure zones
plot_top_failure_zones(analysis['spatial'], config)

# Display detailed zone failure statistics
for model_name, zone_stats in analysis['spatial'].items():
    print(f'\n{model_name} - Top 10 Failure Zones:')
    cols = ['failure_rate', 'mean_abs_error', 'mean_demand']
    if 'Zone' in zone_stats.columns:
        cols = ['Zone', 'Borough'] + cols
    print(zone_stats[cols].head(10).to_string())

In [ ]:
# Borough-level failure analysis
if zone_lookup is not None:
    for model_name, errors_df in analysis['errors'].items():
        errors_with_borough = errors_df.merge(
            zone_lookup[['LocationID', 'Borough']], 
            left_on='zone_id', right_on='LocationID', how='left'
        )
        
        borough_stats = errors_with_borough.groupby('Borough').agg(
            total=('is_failure', 'count'),
            failures=('is_failure', 'sum'),
            mean_error=('abs_error', 'mean')
        ).reset_index()
        borough_stats['failure_rate'] = (borough_stats['failures'] / borough_stats['total'] * 100).round(2)
        borough_stats = borough_stats.sort_values('failure_rate', ascending=False)
        
        print(f'\n{model_name} - Failure Rate by Borough:')
        print(borough_stats.to_string(index=False))

## 5. Failure by Demand Level

In [ ]:
# Failure rate across demand bins
plot_demand_vs_error(analysis['demand_level'], config)

for model_name, demand_analysis in analysis['demand_level'].items():
    print(f'\n{model_name} - Failure Rate by Demand Level:')
    print(demand_analysis.to_string(index=False))

## 6. Cross-Model Failure Comparison

In [ ]:
# Which failures are common across all models?
if analysis.get('cross_model'):
    cm = analysis['cross_model']
    
    print('Cross-Model Failure Analysis:')
    for model, fset in cm['failure_sets'].items():
        print(f'  {model}: {len(fset):,} failure points')
    print(f'  Common failures (all models): {len(cm["common_failures"]):,}')
    print(f'  Total unique failures: {len(cm["total_unique"]):,}')
    print(f'  Overlap: {len(cm["common_failures"])/len(cm["total_unique"])*100:.1f}%')
    
    # Analyze common failures
    if len(cm['common_failures']) > 0:
        common_list = list(cm['common_failures'])
        common_zones = [c[0] for c in common_list]
        
        zone_counts = pd.Series(common_zones).value_counts().head(10)
        print(f'\nTop 10 zones in common failures:')
        for zone, count in zone_counts.items():
            print(f'  Zone {zone}: {count} common failure instances')